In [1]:
import os
import nest_asyncio
nest_asyncio.apply()
import asyncio
from azure.eventhub import EventData
from azure.eventhub.aio import EventHubProducerClient

In [3]:
async def publish_event_to_eventhub(connection_string, eventhub_name, event_data_payload: str):
    """
    Publishes a single event to Azure Event Hub without using an explicit batch.

    Args:
        event_data_payload: The string content of the event to send.
        :param event_data_payload:
        :param connection_string:
        :param eventhub_name:
    """
    if not connection_string or connection_string == "":
        print("Error: Please set your EVENT_HUB_CONNECTION_STRING in the script.")
        print("You can get this from your Azure Event Hub Shared Access Policies.")
        return

    print(f"Attempting to publish event to Event Hub '{eventhub_name}'...")
    producer = None # Initialize producer outside try block

    try:
        # Create a producer client to send events to the Event Hubs namespace
        producer = EventHubProducerClient.from_connection_string(
            conn_str=connection_string,
            eventhub_name=eventhub_name
        )
        print("Event Hub Producer Client created.")
        async with producer:
            # Create an EventData object
            event_batch_data = await producer.create_batch()
            event_data = EventData(event_data_payload)
            event_batch_data.add(event_data)
            print(f"Sending individual event data: '{event_data_payload}'.")
            # Send the single event to the Event Hub
            await producer.send_batch(event_batch_data)
            print(f"Successfully published event: '{event_data_payload}' to '{eventhub_name}'.")
        return "OK"
    except Exception as e:
        print(f"An error occurred while publishing the event: {e}")
        print("Please ensure your connection string, Event Hub name, and permissions are correct.")
    finally:
        if producer:
            # Although 'async with producer' handles closing, an explicit close can be added if needed
            # For simplicity, we rely on 'async with' here.
            pass

In [12]:
sample_event_message = "Hello from Python Event Hub Publisher!"
print(f"Sending a sample event: '{sample_event_message}'")
response = asyncio.run(publish_event_to_eventhub("Endpoint=sb://localhost;SharedAccessKeyName=RootManageSharedAccessKey;SharedAccessKey=SAS_KEY_VALUE;UseDevelopmentEmulator=true","eh1",sample_event_message))
print("event published successfully", response)

Sending a sample event: 'Hello from Python Event Hub Publisher!'
Attempting to publish event to Event Hub 'eh1'...
Event Hub Producer Client created.
Sending individual event data: 'Hello from Python Event Hub Publisher!'.
Successfully published event: 'Hello from Python Event Hub Publisher!' to 'eh1'.
event published successfully OK
